![Banner](https://raw.githubusercontent.com/ralf-42/Image/main/genai-banner-2.jpg)

<p><font size="6" color='grey'> <b>

KI-Agenten. Verstehen. Anwenden. Gestalten.

</b></font> </br></p>

<p><font size="5" color='grey'> <b>
M08 | RAG-Konzepte &amp; Embeddings
</b></font> </br></p>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]    = "M08-RAG-Konzepte"
os.environ["LANGSMITH_ENDPOINT"]   = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment, get_ipinfo, setup_api_keys,
    mprint, install_packages, mermaid,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

In [ ]:
#@title 📦 Pakete installieren{ display-mode: "form" }

install_packages([
                ('markitdown[all]', 'markitdown'),
                'langchain_huggingface',
                ('unstructured[all-docs]>=0.11.2', 'unstructured'),
                ])

# 1 | Übersicht
---

**RAG** (**R**etrieval-**A**ugmented **G**eneration) erweitert Sprachmodelle um **externes Wissen**.  
Ein Agent kann damit Informationen abrufen, die nicht im Training enthalten waren –  
z. B. aktuelle Daten, firmeneigene Dokumente oder spezialisiertes Fachwissen.

## Vergleich: LLM vs. LLM + RAG

| Eigenschaft | LLM allein | LLM + RAG |
|-------------|-----------|----------|
| Wissensbasis | Bis Trainings-Cutoff | Ständig aktualisierbar |
| Private Daten | ❌ Nicht vorhanden | ✅ Aus Vektordatenbank |
| Halluzinationen | Häufig bei unbekanntem Wissen | Reduziert durch Quellenbindung |
| Anpassbarkeit | Fine-Tuning notwendig | Neue Dokumente einfach hinzufügen |
| Transparenz | Schwer nachvollziehbar | Quellen sichtbar |

## RAG als Grundlage für intelligente Agenten

Ein RAG-Agent entscheidet **selbst**, wann er externes Wissen benötigt – und ruft dann gezielt Informationen aus einer Wissensdatenbank ab.

<img src="https://raw.githubusercontent.com/ralf-42/Agenten/main/07_image/rag_process.png" width="700" alt="Avatar">

*RAG-Prozess: Von der Indexierung bis zur Antwort-Generierung*

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  flowchart</font> </br></p>

diagram = '''
flowchart LR
    subgraph Indexierung
        D["📄 Dokumente"] --> C["✂️ Chunking"]
        C --> E["🔢 Embeddings"]
        E --> V[("🗄️ Vektordatenbank")]
    end
    subgraph Retrieval
        Q["❓ Anfrage"] --> QE["🔢 Query-Embedding"]
        QE --> S["🔍 Similarity Search"]
        V --> S
        S --> R["📋 Top-K Dokumente"]
    end
    subgraph Generation
        R --> P["📝 Angereicherter Prompt"]
        Q --> P
        P --> L["🤖 LLM"]
        L --> A["💬 Antwort"]
    end
'''

mermaid(diagram, width=1100)

# 2 | Warum RAG?
---

## Das Problem: LLMs sind eingefroren

Sprachmodelle werden mit Daten bis zu einem bestimmten Zeitpunkt trainiert.  
Danach wissen sie nichts über:

- 📅 **Neue Ereignisse** – Aktuelles aus Wirtschaft, Technik, Wissenschaft
- 🔒 **Vertrauliche Daten** – Interne Dokumente, Kundendaten, proprietäres Wissen
- 📚 **Spezialisiertes Fachwissen** – Produktdokumentation, Rechtsnormen, Handbücher

## Die RAG-Lösung

RAG trennt das **Reasoning** (im LLM) vom **Wissen** (in der Datenbank):

1. 🔍 **Retrieval** – Relevante Dokumente aus einer Wissensbasis abrufen
2. 🔧 **Augmentation** – Den LLM-Prompt mit diesen Dokumenten anreichern
3. ✍️ **Generation** – Das LLM antwortet basierend auf den abgerufenen Informationen

## Wann RAG verwenden?

| Szenario | RAG? | Begründung |
|----------|------|------------|
| FAQ für eigene Produkte | ✅ | Private, sich ändernde Daten |
| Allgemeine Wissensfragen | ❌ | LLM-Basiswissen genügt |
| Interne Dokumentensuche | ✅ | Vertrauliche Unternehmensdaten |
| Mathematik / Code | ❌ | Tools sind effizienter |
| Aktuelle Nachrichten | ✅ | Übersteigt Trainings-Cutoff |

# 3 | RAG-Architektur
---

RAG besteht aus **zwei Phasen**:

**Phase 1 – Indexierung** (einmalig, offline):  
Dokumente werden in Chunks aufgeteilt, in Vektoren umgewandelt und in einer Datenbank gespeichert.

**Phase 2 – Retrieval & Generation** (bei jeder Anfrage):  
Die Nutzeranfrage wird in einen Vektor umgewandelt, die ähnlichsten Chunks gefunden,
und der LLM antwortet mit diesem Kontext.

Durch diese Trennung lässt sich das Wissen **unabhängig vom LLM aktualisieren**.

<img src="https://raw.githubusercontent.com/ralf-42/Agenten/main/07_image/rag_process_01.png" width="650" alt="Avatar">

*Phase 1 – Indexierung: Dokumente → Chunks → Embeddings → Vektordatenbank*

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  flowchart</font> </br></p>

diagram = '''
flowchart TB
    subgraph Phase1["Phase 1: Indexierung (offline)"]
        direction LR
        DOC["📄 Dokument"] --> SPLIT["✂️ Text Splitter"]
        SPLIT --> EMB["🔢 Embedding Model"]
        EMB --> DB[("🗄️ ChromaDB")]
    end

    subgraph Phase2["Phase 2: Retrieval und Generation (online)"]
        direction LR
        USER["👤 Nutzer"] --> QEMB["🔢 Query Embedding"]
        QEMB --> SEARCH["🔍 Similarity Search"]
        DB --> SEARCH
        SEARCH --> PROMPT["📝 Prompt Builder"]
        USER --> PROMPT
        PROMPT --> LLM["🤖 LLM"]
        LLM --> ANS["💬 Antwort"]
    end

    Phase1 -.->|"Einmalig aufgebaut"| Phase2
'''

mermaid(diagram, width=600)

# 4 | Embeddings erklärt
---

## Was sind Embeddings?

Ein **Embedding** ist eine numerische Darstellung von Text als Vektor in einem hochdimensionalen Raum.  
Semantisch ähnliche Texte liegen dabei **nah beieinander** im Vektorraum.

```
"Hund spielt"  →  [0.23, -0.11,  0.87, ...]   <- nah beieinander!
"Tier tobt"    →  [0.21, -0.09,  0.84, ...]
"Aktienkurs"   →  [-0.45, 0.72, -0.31, ...]   <- weit entfernt
```

## OpenAI Embedding-Modelle

| Modell | Dimensionen | Kosten | Empfehlung |
|--------|------------|--------|------------|
| `text-embedding-3-small` | 1.536 | Günstig | ✅ Standard |
| `text-embedding-3-large` | 3.072 | Höher | Anspruchsvolle Suche |
| `text-embedding-ada-002` | 1.536 | Mittel | ⚠️ Veraltet |

## Ähnlichkeitsmaße

| Maß | Wertebereich | Empfehlung |
|-----|------------|------------|
| **Kosinus-Ähnlichkeit** | -1 bis 1 | ✅ OpenAI empfohlen |
| Euklidischer Abstand | 0 bis ∞ | Alternativ |

**Interpretation:** 1.0 = identisch  •  0.5 = verwandt  •  0.0 = unrelated  •  -1.0 = gegensätzlich

# 5 | Embedding-Demo
---

Wir erstellen Embeddings für drei Sätze und vergleichen ihre semantische Ähnlichkeit:

- **Satz A & B** – Beide über Tiere → sollten **ähnlich** sein
- **Satz A & C** – Tier vs. Börse → sollten **unähnlich** sein

In [ ]:
from langchain_openai import OpenAIEmbeddings
import numpy as np

embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")
print("✅ Embedding-Modell geladen: text-embedding-3-small (1536 Dimensionen)")

In [ ]:
# Drei Beispielsätze
satz_a = "Der Hund spielt im Park."
satz_b = "Das Tier tobt auf der Wiese."
satz_c = "Die Aktie fiel gestern um 3 Prozent."

vektoren = embeddings_model.embed_documents([satz_a, satz_b, satz_c])

def cosine_similarity(v1, v2):
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

sim_ab = cosine_similarity(vektoren[0], vektoren[1])
sim_ac = cosine_similarity(vektoren[0], vektoren[2])

mprint(f"""
## Ergebnis: Semantische Ähnlichkeit

| Vergleich | Thema | Ähnlichkeit | Bewertung |
|-----------|-------|-------------|----------|
| A vs. B | Tier-Themen | **{sim_ab:.4f}** | 🟢 Ähnlich |
| A vs. C | Tier vs. Börse | **{sim_ac:.4f}** | 🔴 Unähnlich |

Vektordimensionen: {len(vektoren[0])}
""")

# 6 | Chunking-Strategien
---

## Warum Chunking?

LLMs haben ein begrenztes Kontextfenster.  
Lange Dokumente müssen in **Chunks** aufgeteilt werden, bevor sie indexiert werden können.

## Strategien im Vergleich

| Strategie | Beschreibung | Vorteil | Nachteil |
|-----------|-------------|---------|----------|
| **Fixed-Size** | Feste Zeichenanzahl | Einfach, schnell | Zerreißt Sätze |
| **Sentence-based** | An Satzgrenzen | Semantisch sauber | Variable Größe |
| **Recursive** | Hierarchisch: Absatz → Satz → Wort | ✅ Ausgewogen | Komplexer |
| **Semantic** | Nach semantischen Gruppen | Beste Qualität | Langsam, teuer |

**Empfehlung:** `RecursiveCharacterTextSplitter` mit `chunk_size=500, chunk_overlap=50`

> **Faustregel:** 1 DIN-A4-Seite ≈ 300 Wörter ≈ 450 Tokens

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = (
    "Kuenstliche Intelligenz veraendert die Arbeitswelt grundlegend. "
    "Viele Routineaufgaben werden automatisiert, waehrend kreative Taetigkeiten wichtiger werden.\n\n"
    "Agenten-Systeme sind besonders leistungsstark, weil sie eigenstaendig Entscheidungen treffen koennen. "
    "Ein Agent kann Tools aufrufen, Informationen abrufen und komplexe Aufgaben zerlegen.\n\n"
    "RAG-Systeme ergaenzen diese Faehigkeiten um externes Wissen. "
    "Durch Agenten und RAG entstehen Systeme, die sowohl denken als auch nachschlagen koennen."
)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30,
    length_function=len,
)

chunks = splitter.split_text(text)

print(f"Original:  {len(text)} Zeichen")
print(f"Chunks:    {len(chunks)} Stueck (chunk_size=200, overlap=30)\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({len(chunk)} Zeichen) ---")
    print(chunk)
    print()

# 7 | Token-Limits & Context Window
---

Das Kontextfenster muss für **drei Komponenten** reichen:

```
Context Window (z.B. 128k Tokens bei GPT-4o)
┌──────────────┬──────────────────┬──────────────┐
│ System-Prompt│  Top-K Chunks    │  Antwort     │
│  ~300 Tokens │  K x 400 Tokens  │  ~500 Tokens │
└──────────────┴──────────────────┴──────────────┘
```

## Praxisregeln

| Parameter | Empfehlung | Begründung |
|-----------|-----------|------------|
| `chunk_size` | 300–500 Zeichen | Kompakte, fokussierte Chunks |
| `chunk_overlap` | 10–15% von chunk_size | Verhindert Informationsverlust an Grenzen |
| Top-K | 3–5 Chunks | Meist ausreichend, spart Tokens |
| Embedding-Modell | `text-embedding-3-small` | Gutes Preis-Leistungs-Verhältnis |

- Warum Chunk-Größe und Overlap Kosten und Qualität beeinflussen
- Prompt-Budget: Frage + Kontext + Antwortfenster vorausplanen
- Retrieval-Tiefe nur so groß wie nötig

# 8 | Mini-RAG: Vom Chunk zur Antwort
---

Die vorherigen Abschnitte haben Embeddings und Chunking einzeln beleuchtet.  
Jetzt kombinieren wir beides zu einem **vollständigen RAG-Loop** – kompakt, ohne persistente Datenbank.

**Ablauf:**
1. 🗃️ 4 Mini-Dokumente → In-Memory Vektordatenbank  
2. 🔍 Retriever findet die 2 ähnlichsten Chunks  
3. 📝 Prompt wird mit Kontext angereichert  
4. 🤖 LLM antwortet quellgebunden

> **Hinweis:** Das nächste Modul (*ChromaDB & Indexing*) baut dieses Muster aus – mit ChromaDB-Persistenz, Collections und Metadaten-Filtern.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Mini-Wissensdatenbank (4 Sätze, in-memory – kein persist_directory)
mini_docs = [
    'RAG kombiniert Retrieval mit LLM-Generation: Das LLM antwortet basierend auf abgerufenen Dokumenten.',
    'Embeddings sind numerische Vektoren – semantisch ähnliche Texte liegen nah beieinander im Vektorraum.',
    'Chunking teilt lange Dokumente in kleinere Abschnitte, damit sie ins Kontextfenster passen.',
    'ChromaDB ist eine Vektordatenbank, die Embeddings persistent speichert und schnell durchsucht.',
]

vectorstore = Chroma.from_texts(mini_docs, embedding=embeddings_model)
retriever   = vectorstore.as_retriever(search_kwargs={'k': 2})

def format_docs(docs):
    return '\n\n'.join(d.page_content for d in docs)

llm = init_chat_model('openai:gpt-4o-mini', temperature=0.0)

rag_prompt = PromptTemplate.from_template(
    'Beantworte die Frage ausschliesslich auf Basis des Kontexts.\n\n'
    'Kontext:\n{context}\n\n'
    'Frage: {question}\n\n'
    'Antwort:'
)

mini_rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)
print('✅ Mini-RAG-Chain bereit (in-memory, 4 Docs)')

In [ ]:
fragen = [
    'Was ist RAG?',
    'Wozu braucht man Chunking?',
    'Was macht ChromaDB?',
]

for frage in fragen:
    antwort   = mini_rag_chain.invoke(frage)
    retrieved = retriever.invoke(frage)
    mprint(f'''
---
**Frage:** {frage}  
**Quellen:** {len(retrieved)} Chunk(s) abgerufen  
**Antwort:** {antwort}
''')

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M08-RAG-Konzepte", limit=3, show_steps=True)

# A | Aufgabe
---

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

<p><font color='black' size="5">
Embedding-Vergleich: Thematische Cluster
</font></p>

Erstellen Sie Embeddings für einen kleinen Textkorpus (6–9 Sätze aus 3 verschiedenen Themen)
und untersuchen Sie, ob die Ähnlichkeitsmaße die Themengruppen korrekt erkennen.

**Teilaufgaben:**
1. Wählen Sie 3 Themen (z.B. Sport, Technik, Kochen) mit je 2–3 Sätzen
2. Erstellen Sie die Embeddings mit `text-embedding-3-small`
3. Berechnen Sie die Kosinus-Ähnlichkeit für alle Paare
4. Stellen Sie die Ergebnisse in einer Markdown-Tabelle dar

**Bonus:** Vergleichen Sie `text-embedding-3-small` mit `text-embedding-3-large` –
gibt es Unterschiede bei der Clustererkennung?